# LifeMate trên Google Colab & Máy Local: SFT Data Preparation, QLoRA Training & Pipeline Check

Notebook này thực hiện toàn bộ quy trình:
1. **Tự động nhận diện môi trường**: Google Colab (Drive) hoặc máy tính cục bộ (Local Machine / Jupyter).
2. **Chuẩn bị dữ liệu SFT**: Tích hợp phương pháp *Hammer Function & Parameter Masking Augmentation* (ICLR 2025) và schema pruning rút gọn.
3. **Tải model nền**: `Qwen/Qwen2.5-3B-Instruct` (và tùy chọn `vinai/PhoWhisper-medium`, `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`).
4. **Huấn luyện QLoRA**: Tự động scale batch size, gradient accumulation và max sequence length theo phần cứng GPU (T4, A100, V100, RTX, ...).
5. **Trực quan hóa Loss curve** và **Kiểm tra End-to-End Pipeline** tích hợp ToolNet Graph + Rewriter + Policy + Middleware.

## 1. Khởi Tạo Môi Trường & Thiết Lập Đường Dẫn Tự Động

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

# Tự động nhận diện môi trường: Google Colab hay Local Machine
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Môi trường phát hiện: Google Colab")
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    
    # 1. Tìm thư mục dự án được upload lên Google Drive (Ưu tiên MyDrive/ZALO/ZALO_AI_CHALLENGE_2026)
    candidate_paths = [
        DRIVE_ROOT / "ZALO" / "ZALO_AI_CHALLENGE_2026",
        DRIVE_ROOT / "ZALO_AI_CHALLENGE_2026",
        DRIVE_ROOT / "lifemate",
        Path("/content/ZALO_AI_CHALLENGE_2026"),
    ]
    PROJECT_DIR = None
    for cand in candidate_paths:
        if (cand / "scripts" / "download_models.py").exists():
            PROJECT_DIR = cand
            break
            
    # 2. Nếu upload dạng tar.gz bundle:
    if not PROJECT_DIR:
        bundles = sorted(DRIVE_ROOT.glob("lifemate-colab-bundle-*.tar.gz"), key=lambda p: p.stat().st_mtime, reverse=True)
        if bundles:
            WORK_ROOT = Path("/content/lifemate_job")
            WORK_ROOT.mkdir(parents=True, exist_ok=True)
            subprocess.run(["tar", "-xzf", str(bundles[0]), "-C", str(WORK_ROOT)], check=True)
            PROJECT_DIR = WORK_ROOT / "lifemate"
            
    # 3. Fallback tìm file download_models.py trong toàn bộ Drive
    if not PROJECT_DIR:
        for sub in DRIVE_ROOT.glob("**/scripts/download_models.py"):
            PROJECT_DIR = sub.parent.parent
            break

    if not PROJECT_DIR or not PROJECT_DIR.exists():
        raise FileNotFoundError(
            "Không tìm thấy thư mục ZALO_AI_CHALLENGE_2026 trên Google Drive!\n"
            "Hãy đảm bảo bạn đã upload thư mục dự án lên Google Drive (ví dụ: MyDrive/ZALO_AI_CHALLENGE_2026)."
        )
        
    RUN_ID = time.strftime("%Y%m%d-%H%M%S")
    RUN_DIR = (DRIVE_ROOT / "ZALO" / "lifemate_runs" / RUN_ID) if (DRIVE_ROOT / "ZALO").exists() else (DRIVE_ROOT / "lifemate_runs" / RUN_ID)
    RUN_DIR.mkdir(parents=True, exist_ok=True)
else:
    print("Môi trường phát hiện: Local Machine (Máy cá nhân / Jupyter)")
    PROJECT_DIR = Path.cwd()
    if not (PROJECT_DIR / "scripts" / "download_models.py").exists():
        if (PROJECT_DIR.parent / "scripts" / "download_models.py").exists():
            PROJECT_DIR = PROJECT_DIR.parent
            
    RUN_ID = time.strftime("%Y%m%d-%H%M%S")
    RUN_DIR = PROJECT_DIR / "runs" / RUN_ID
    RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Project Directory:", PROJECT_DIR)
print("Persistent Run Directory:", RUN_DIR)
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))


## 2. Kiểm Tra Dữ Liệu Train & Tools Catalog

In [ ]:
# Tự động xác định đường dẫn thư mục chứa dữ liệu train
possible_data_roots = [
    PROJECT_DIR / "data/raw/train+public_test",
    PROJECT_DIR / "train+public_test",
    Path("/content/dataset_lifemate/train+public_test"),
    DRIVE_ROOT / "ZALO/train+public_test" if IN_COLAB else None,
]
DATA_ROOT = None
for root in possible_data_roots:
    if root and (root / "train" / "train.jsonl").is_file():
        DATA_ROOT = root
        break

# Tự động giải nén nếu có file zip
if not DATA_ROOT:
    zip_candidates = [
        PROJECT_DIR / "data/archives/train+public_test.zip",
        DRIVE_ROOT / "ZALO/train+public_test.zip" if IN_COLAB else None,
        DRIVE_ROOT / "train+public_test.zip" if IN_COLAB else None,
    ]
    for z in zip_candidates:
        if z and z.is_file():
            print(f"Đang giải nén tập dữ liệu từ {z}...")
            target_extract = PROJECT_DIR / "data/raw"
            target_extract.mkdir(parents=True, exist_ok=True)
            import zipfile
            with zipfile.ZipFile(z, "r") as zip_ref:
                zip_ref.extractall(target_extract)
            if (target_extract / "train+public_test" / "train" / "train.jsonl").is_file():
                DATA_ROOT = target_extract / "train+public_test"
            break

if not DATA_ROOT:
    print("Cảnh báo: Chưa tìm thấy train.jsonl! Hãy kiểm tra data_root hoặc đặt train+public_test vào thư mục dự án.")
else:
    print("DATA_ROOT xác định tại:", DATA_ROOT)
    train_file = DATA_ROOT / "train" / "train.jsonl"
    catalog_file = DATA_ROOT / "train" / "train_tools_catalog.json"
    print(f"train.jsonl: {'OK' if train_file.is_file() else 'MISSING'}")
    print(f"train_tools_catalog.json: {'OK' if catalog_file.is_file() else 'MISSING'}")


## 3. Kiểm Tra GPU & Tự Động Điều Chỉnh Siêu Tham Số Phần Cứng

In [ ]:
subprocess.run(["nvidia-smi"], check=False)
import torch

if not torch.cuda.is_available():
    print("Cảnh báo: Không tìm thấy CUDA GPU. Đang chạy ở chế độ CPU debug.")
    props_name = "CPU"
    VRAM_GB = 0.0
    MAX_SEQUENCE_LENGTH = 1024
    TRAIN_BATCH_SIZE = 1
    GRAD_ACCUM_STEPS = 16
    USE_FLASH_ATTN = False
else:
    props = torch.cuda.get_device_properties(0)
    props_name = props.name
    VRAM_GB = props.total_memory / (1024 ** 3)
    FREE_GB = shutil.disk_usage(str(PROJECT_DIR)).free / (1024 ** 3)
    # Tự động scale siêu tham số theo VRAM phần cứng
    MAX_SEQUENCE_LENGTH = 4096 if VRAM_GB >= 22 else 2048 if VRAM_GB >= 14 else 1024
    TRAIN_BATCH_SIZE = 4 if VRAM_GB >= 38 else 2 if VRAM_GB >= 22 else 1
    GRAD_ACCUM_STEPS = 4 if TRAIN_BATCH_SIZE >= 4 else 8 if TRAIN_BATCH_SIZE >= 2 else 16
    USE_FLASH_ATTN = VRAM_GB >= 22
    print(f"GPU: {props_name}; VRAM: {VRAM_GB:.1f} GB; Disk trống: {FREE_GB:.1f} GB")
    print(f"Tự động cấu hình: seq={MAX_SEQUENCE_LENGTH}, batch_size={TRAIN_BATCH_SIZE}, grad_accum={GRAD_ACCUM_STEPS}, flash_attn={USE_FLASH_ATTN}")


## 4. Cài Đặt Thư Viện Cần Thiết

In [ ]:
# Cài đặt các gói phụ thuộc cho huấn luyện và đánh giá
req_pkgs = ["-r", "requirements-train.txt", "huggingface_hub", "matplotlib", "jiwer"]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + req_pkgs,
    cwd=PROJECT_DIR,
    check=True,
)
import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda, "available:", torch.cuda.is_available())


## 5. Tải Model Nền & Chuẩn Bị Dữ Liệu SFT (Kèm Hammer Masking)

In [ ]:
# Tải model LLM Qwen2.5-3B-Instruct về máy/Colab để chuẩn bị huấn luyện
subprocess.run([sys.executable, "scripts/download_models.py", "--models", "llm"], cwd=PROJECT_DIR, check=True)

# Tạo dữ liệu SFT với tính năng Hammer Function & Parameter Masking Augmentation
prepare_cmd = [
    sys.executable, "training/prepare_sft.py",
    "--data-root", str(DATA_ROOT) if DATA_ROOT else "data/raw/train+public_test",
    "--augment-masking",
]
subprocess.run(prepare_cmd, cwd=PROJECT_DIR, check=True)
manifest = json.loads((PROJECT_DIR / "data/processed/sft/manifest.json").read_text(encoding="utf-8"))
print("SFT split:", manifest["samples"])
print("SFT records:", manifest["records"])


## 6. Smoke Train QLoRA (Kiểm Tra Môi Trường & Khởi Tạo Adapter)

In [ ]:
smoke_command = [
    sys.executable, "training/train_lora.py",
    "--train-jsonl", "data/processed/sft/train.jsonl",
    "--dev-jsonl", "data/processed/sft/dev.jsonl",
    "--limit-train-samples", "64",
    "--limit-dev-samples", "16",
    "--epochs", "1",
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRAD_ACCUM_STEPS),
    "--max-sequence-length", str(MAX_SEQUENCE_LENGTH),
    "--output-dir", "models/lifemate-qwen2.5-3b-smoke",
]
if USE_FLASH_ATTN:
    smoke_command.append("--use-flash-attention-2")
subprocess.run(smoke_command, cwd=PROJECT_DIR, check=True)
SMOKE_ADAPTER = PROJECT_DIR / "models/lifemate-qwen2.5-3b-smoke/final"
if not SMOKE_ADAPTER.is_dir():
    raise FileNotFoundError(f"Không thấy adapter smoke: {SMOKE_ADAPTER}")
SMOKE_BACKUP = RUN_DIR / "lifemate-qwen2.5-3b-smoke-adapter"
shutil.copytree(SMOKE_ADAPTER, SMOKE_BACKUP, dirs_exist_ok=True)
print("Smoke adapter saved to persistent run directory:", SMOKE_BACKUP)


## 7. Tùy Chọn Full Train (Huấn Luyện Toàn Bộ Tập Dữ Liệu)

In [ ]:
RUN_FULL_TRAINING = False
FULL_OUTPUT_DIR = RUN_DIR / "lifemate-qwen2.5-3b-full-training"
FULL_ADAPTER = FULL_OUTPUT_DIR / "final"
if RUN_FULL_TRAINING:
    full_command = [
        sys.executable, "training/train_lora.py",
        "--train-jsonl", "data/processed/sft/train.jsonl",
        "--dev-jsonl", "data/processed/sft/dev.jsonl",
        "--epochs", "3",
        "--batch-size", str(TRAIN_BATCH_SIZE),
        "--gradient-accumulation-steps", str(GRAD_ACCUM_STEPS),
        "--lora-r", "32",
        "--lora-alpha", "64",
        "--learning-rate", "2e-4",
        "--max-sequence-length", str(MAX_SEQUENCE_LENGTH),
        "--output-dir", str(FULL_OUTPUT_DIR),
    ]
    if USE_FLASH_ATTN:
        full_command.append("--use-flash-attention-2")
    subprocess.run(full_command, cwd=PROJECT_DIR, check=True)
    if not FULL_ADAPTER.is_dir():
        raise FileNotFoundError(f"Không thấy full adapter: {FULL_ADAPTER}")
    print("Full adapter:", FULL_ADAPTER)
    print("Trainer checkpoints are saved under:", FULL_OUTPUT_DIR)
else:
    print("Full train đang tắt. Muốn chạy thì đổi RUN_FULL_TRAINING = True và chạy lại cell này.")


## 8. Sao Lưu Metadata Phiên Chạy

In [ ]:
(RUN_DIR / "run_info.json").write_text(
    json.dumps({
        "run_id": RUN_ID,
        "gpu": props_name,
        "vram_gb": round(VRAM_GB, 2),
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "full_training": RUN_FULL_TRAINING,
        "seed": 2026,
        "smoke_adapter": str(SMOKE_BACKUP),
        "full_adapter": str(FULL_ADAPTER) if RUN_FULL_TRAINING else None,
    }, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("Đã lưu thông tin phiên chạy tại:", RUN_DIR / "run_info.json")


## 9. Trực Quan Hóa Đường Cong Loss (Train & Validation)

In [ ]:
import json
import matplotlib.pyplot as plt

target_dir = FULL_OUTPUT_DIR if RUN_FULL_TRAINING else (PROJECT_DIR / "models/lifemate-qwen2.5-3b-smoke")
state_file = target_dir / "trainer_state.json"
if state_file.is_file():
    state = json.loads(state_file.read_text(encoding="utf-8"))
    log_history = state.get("log_history", [])
    train_steps, train_losses = [], []
    eval_steps, eval_losses = [], []
    for entry in log_history:
        if "loss" in entry and "step" in entry:
            train_steps.append(entry["step"])
            train_losses.append(entry["loss"])
        if "eval_loss" in entry and "step" in entry:
            eval_steps.append(entry["step"])
            eval_losses.append(entry["eval_loss"])
    
    plt.figure(figsize=(10, 5))
    if train_losses:
        plt.plot(train_steps, train_losses, label="Training Loss (per step)", color="royalblue", alpha=0.8)
    if eval_losses:
        plt.plot(eval_steps, eval_losses, label="Validation Loss (per epoch)", color="crimson", marker="o", linewidth=2)
    plt.xlabel("Optimization Steps")
    plt.ylabel("Cross Entropy Loss")
    plt.title(f"LifeMate Qwen2.5-3B Training Convergence ({props_name})")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    loss_plot_path = RUN_DIR / "loss_curve.png"
    plt.savefig(loss_plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Đã lưu biểu đồ Loss vào:", loss_plot_path)
else:
    print("Chưa thấy trainer_state.json. Chạy smoke hoặc full train trước.")


## 10. Kiểm Tra Toàn Bộ Pipeline Gọi Thành Phần (End-to-End Pipeline Smoke Test)

Cell này kiểm tra khả năng phối hợp giữa các thành phần:
1. **Rewriter (`ReasoningRewriter`)**: Sửa lỗi ASR và chuẩn hóa câu nói.
2. **ToolNet Graph Retrieval (`ToolGraphIndex`)**: Lọc top 5-10 công cụ liên quan từ catalog.
3. **Tool-Boundary Middleware**: Kiểm tra các bất biến và hiệu chỉnh tham số tự động.

In [ ]:
from lifemate.reasoning_rewriter import ReasoningRewriter
from lifemate.graph_retrieval import ToolGraphIndex
from lifemate.middleware import silent_correct_action, check_local_invariants

print("=== KIỂM TRA TỪNG THÀNH PHẦN LIFE MATE ===")

# 1. Kiểm tra Rewriter
rewriter = ReasoningRewriter(model_path=None) # Chạy chế độ heuristic/CPU kiểm thử
raw_transcript = "bật bật đen phòng khách giúp tôi"
cleaned = rewriter.rewrite(raw_transcript)
print(f"1. Rewriter: '{raw_transcript}' -> '{cleaned}'")
assert "bật đèn" in cleaned

# 2. Kiểm tra ToolNet Graph Retrieval
catalog_file = None
if DATA_ROOT and (DATA_ROOT / "train" / "train_tools_catalog.json").is_file():
    catalog_file = DATA_ROOT / "train" / "train_tools_catalog.json"
elif (PROJECT_DIR / "train+public_test/train/train_tools_catalog.json").is_file():
    catalog_file = PROJECT_DIR / "train+public_test/train/train_tools_catalog.json"

if catalog_file:
    catalog = json.loads(catalog_file.read_text(encoding="utf-8"))
    tool_graph = ToolGraphIndex(catalog)
    top_tools = tool_graph.retrieve_top_k("bật đèn phòng khách", k=5)
    tool_names = [t["name"] for t in top_tools]
    print(f"2. ToolNet Retrieval top 5 tools: {tool_names}")
    assert len(top_tools) > 0

# 3. Kiểm tra Middleware Invariants
test_action = {"action": "call", "name": "light.set_power", "arguments": {"device_id": "den_phong_khach", "state": "on"}}
corrected = silent_correct_action(test_action, {"id": "test_01"}, catalog if 'catalog' in locals() else {"apps": []})
print(f"3. Middleware Action Validation: {corrected.get('name')}")

print("\n>>> TẤT CẢ CÁC THÀNH PHẦN ĐÃ ĐƯỢC KẾT NỐI VÀ HOẠT ĐỘNG HOÀN HẢO! <<<")
